# Traffic benchmark: 1. Construct the reference outcome table

This notebook constructs the versioned artificial-gap benchmark for the Traffic domain. It implements the frozen reference protocol, writes derived outputs only, and never copies external source data into the repository.

**Purpose.** The resulting table combines observable local context, per-method outcomes, and gap provenance. It is the shared input to the descriptive analysis, confirmatory nested evaluation, and final deployment fit in notebooks 2–4.


## Data scope and eligibility

The frozen LargeST/PeMS reference includes five-minute flow from Districts 3, 4, 7, and 11 for 2017–2021. It retains sensors with at least 95% valid coverage and no natural gap longer than 24 hours; non-finite and negative values are natural gaps, while zero flow is valid.

For each district-year, 100 sensor-years are selected deterministically. Each contributes one non-overlapping fully observed gap in five strata (1–3, 4–12, 13–36, 37–144, and 145–288 five-minute steps), requesting 10,000 gaps.


## Reproducible inputs and deliberate rebuilds

`TRAFFIC_DATA_DIR` in `.env` must point to the required external data root. `TRAFFIC_BENCHMARK_DIR` can optionally redirect the derived benchmark output; the published default is `benchmarks/traffic`.

The frozen protocol is defined in `configs/traffic_final.toml`. It fixes sampling, context requirements, duration strata, random seed, and the scale-floor quantile. Set `PREPARE_PANEL = True` only to deliberately audit the external raw archive and rebuild the selected panel. Otherwise the existing audited panel under the external data root is reused.


In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from gap_imputation_benchmark.paths import load_local_environment, local_path_or_default, project_relative_path_or_label
load_local_environment(override=True)

if not os.environ.get('TRAFFIC_DATA_DIR'):
    raise RuntimeError("Set TRAFFIC_DATA_DIR in .env or in the current session.")

DATA_DIR = Path(os.environ['TRAFFIC_DATA_DIR'])
BENCHMARK_DIR = local_path_or_default(
    'TRAFFIC_BENCHMARK_DIR', PROJECT_ROOT / 'benchmarks' / 'traffic',
)
CONFIG = PROJECT_ROOT / 'configs' / 'traffic_final.toml'
SCRIPT = PROJECT_ROOT / 'scripts' / 'build_traffic_benchmark.py'
PREPARE_PANEL = False

BENCHMARK_LOCATION = project_relative_path_or_label(
    BENCHMARK_DIR, fallback_label='configured benchmark directory'
)
print('External data: configured external data directory')
print(f'Benchmark output: {BENCHMARK_LOCATION}')


External data: configured external data directory
Benchmark output: benchmarks/traffic


## Generate the benchmark

The canonical script evaluates the same artificial-gap geometry for every registered candidate method: forward fill, nearest boundary, linear interpolation, PCHIP, local natural cubic spline, BIC-selected polynomial, template reconstruction, and a bidirectional weekly seasonal reference. The seasonal reference searches weekly offsets in both directions, requires at least three complete references, and aggregates them pointwise by the median. Each gap retains portable source references and diagnostics needed for review.


In [2]:
import subprocess

execution_env = os.environ.copy()
execution_env['PYTHONPATH'] = source_root + os.pathsep + execution_env.get('PYTHONPATH', '')
command = [
    sys.executable, str(SCRIPT), '--config', str(CONFIG),
    '--data-dir', str(DATA_DIR), '--output-dir', str(BENCHMARK_DIR),
]
if PREPARE_PANEL:
    command.append('--prepare-panel')
subprocess.run(command, cwd=PROJECT_ROOT, env=execution_env, check=True)
print(f'Benchmark outputs written to: {BENCHMARK_LOCATION}')


[evaluate] [------------------------] 0/10,000 (0%)
[evaluate] [------------------------] 100/10,000 (1%) | 2.48 gaps/s | ETA 66.4 min
[evaluate] [------------------------] 200/10,000 (2%) | 2.09 gaps/s | ETA 78.0 min
[evaluate] [#-----------------------] 300/10,000 (3%) | 2.04 gaps/s | ETA 79.2 min
[evaluate] [#-----------------------] 400/10,000 (4%) | 2.11 gaps/s | ETA 75.8 min
[evaluate] [#-----------------------] 500/10,000 (5%) | 2.16 gaps/s | ETA 73.5 min
[evaluate] [#-----------------------] 600/10,000 (6%) | 2.28 gaps/s | ETA 68.8 min
[evaluate] [##----------------------] 700/10,000 (7%) | 2.27 gaps/s | ETA 68.3 min
[evaluate] [##----------------------] 800/10,000 (8%) | 2.32 gaps/s | ETA 66.2 min
[evaluate] [##----------------------] 900/10,000 (9%) | 2.34 gaps/s | ETA 64.8 min
[evaluate] [##----------------------] 1,000/10,000 (10%) | 2.33 gaps/s | ETA 64.5 min
[evaluate] [###---------------------] 1,100/10,000 (11%) | 2.33 gaps/s | ETA 63.7 min
[evaluate] [###--------------

## Review the generated reference tables

`coverage_table.csv` verifies requested, learnable, and excluded gaps at the relevant domain level. `selected_recordings.csv` records the deterministically selected source units; `input_manifest.csv` records input discovery and portable references; `metadata.json` records the data scope and frozen protocol.

Any sampling shortfall or post-sampling exclusion remains explicit in the output files. Review these records before treating a rerun as equivalent to the published reference.


In [3]:
import json
import pandas as pd

coverage = pd.read_csv(BENCHMARK_DIR / 'coverage_table.csv')
selected_recordings = pd.read_csv(BENCHMARK_DIR / 'selected_recordings.csv')
metadata = json.loads((BENCHMARK_DIR / 'metadata.json').read_text(encoding='utf-8'))
display(coverage)
display(selected_recordings.head())
metadata


,district,year,duration_stratum,n_gaps_written,n_gaps_requested,shortfall
0,3,2017,1,100,100,0
1,3,2017,2,100,100,0
2,3,2017,3,100,100,0
3,3,2017,4,100,100,0
4,3,2017,5,100,100,0
...,...,...,...,...,...,...
95,11,2021,1,100,100,0
96,11,2021,2,100,100,0
97,11,2021,3,100,100,0
98,11,2021,4,100,100,0


,district,year,sensor_id2,panel_column,n_gaps_written
0,3,2017,1,1,5
1,3,2017,5,5,5
2,3,2017,9,9,5
3,3,2017,12,12,5
4,3,2017,19,19,5


{'artifact_type': 'benchmark',
 'domain': 'traffic',
 'workflow': 'traffic_benchmark',
 'sampling_unit': 'sensor_x_calendar_year',
 'districts': [3, 4, 7, 11],
 'years': [2017, 2018, 2019, 2020, 2021],
 'sensors_per_district_year': 100,
 'gap_strata_steps_inclusive': [[1, 3],
  [4, 12],
  [13, 36],
  [37, 144],
  [145, 288]],
 'interval_minutes': 5,
 'min_context_valid_fraction': 0.8,
 'context_policy': 'max(gap_length_steps, 2) steps per side',
 'max_gap_attempts': 500,
 'random_state': 20260824,
 'method_names': ['forward_fill',
  'nearest_boundary',
  'linear',
  'pchip',
  'local_natural_cubic_spline',
  'polyfit_bic',
  'template',
  'seasonal_periodic'],
 'feature_columns': ['realized_gap_duration_minutes',
  'left_context_valid_fraction',
  'right_context_valid_fraction',
  'normalized_boundary_jump',
  'normalized_mean_difference_right_minus_left',
  'local_std_over_scale',
  'local_range_over_scale',
  'normalized_trend_before',
  'normalized_trend_after',
  'normalized_trend_